# PS S6E6 - 013 Blend add012 lgb010 bias check

Experiment: `exp_20260605_013_blend_add012_lgb010_bias_check`

Purpose:
- Add 012 bias-searched 010 to the 011 blend/correlation table.
- Decide whether raw 010 should be replaced by 012, or whether both raw 010 and biased 012 should remain.
- Check whether 007 and 006 still add value after 012.
- Save only the best blend OOF/pred `.npy` files for the next bias-search experiment.

Important:
- `.npy` files are loaded from `/kaggle/input/datasets/mizushimatoshihiko/npy-files`.
- No training.
- No original dataset.
- No new FE.
- This notebook does not run bias search. It only blends existing OOF/pred probability matrices.

In [1]:
import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier

warnings.filterwarnings("ignore")
try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260605_013_blend_add012_lgb010_bias_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {"key":"002_lgb_raw","exp_id":"exp_20260601_002_lgb_strict_raw_baseline","family":"LightGBM","role":"raw_reference","oof":"oof_lgb_strict_raw_proba.npy","pred":"pred_lgb_strict_raw_proba.npy","cv":0.9569063401002502,"public_lb":0.95800},
    {"key":"003_cat_raw","exp_id":"exp_20260601_003_cat_strict_raw_baseline","family":"CatBoost","role":"weak_reference","oof":"oof_cat_strict_raw_proba.npy","pred":"pred_cat_strict_raw_proba.npy","cv":0.9524581895303758,"public_lb":0.95351},
    {"key":"004_xgb_raw","exp_id":"exp_20260603_004_xgb_strict_raw_baseline","family":"XGBoost","role":"aux_reference","oof":"oof_xgb_strict_raw_proba.npy","pred":"pred_xgb_strict_raw_proba.npy","cv":0.9557836270283273,"public_lb":0.95638},
    {"key":"006_lgb_color_min","exp_id":"exp_20260603_006_lgb_color_index_min","family":"LightGBM","role":"blend_auxiliary","oof":"oof_lgb_color_index_min_proba.npy","pred":"pred_lgb_color_index_min_proba.npy","cv":0.9570463209115138,"public_lb":0.95663},
    {"key":"007_lgb002_bias","exp_id":"exp_20260603_007_lgb002_bias_search","family":"LightGBM","role":"old_mainline_auxiliary","oof":"oof_exp_20260603_007_lgb002_bias_search_proba.npy","pred":"pred_exp_20260603_007_lgb002_bias_search_proba.npy","cv":0.9586098708578418,"public_lb":0.95921},
    {"key":"010_shared002_lgb","exp_id":"exp_20260604_010_shared002_lgb_as_is","family":"LightGBM","role":"mainline_single_raw","oof":"oof_exp_20260604_010_shared002_lgb_as_is_proba.npy","pred":"pred_exp_20260604_010_shared002_lgb_as_is_proba.npy","cv":0.9625129421154345,"public_lb":0.96357},
    {"key":"012_lgb010_bias","exp_id":"exp_20260605_012_lgb010_bias_search","family":"LightGBM","role":"mainline_single_bias","oof":"oof_exp_20260605_012_lgb010_bias_search_proba.npy","pred":"pred_exp_20260605_012_lgb010_bias_search_proba.npy","cv":0.9632282943283176,"public_lb":0.96466},
]

BLEND_SETS = {
    "A_011_best_recheck_010_007_004_006": ["010_shared002_lgb", "007_lgb002_bias", "004_xgb_raw", "006_lgb_color_min"],
    "B_replace_010_with_012_012_007_004_006": ["012_lgb010_bias", "007_lgb002_bias", "004_xgb_raw", "006_lgb_color_min"],
    "C_keep_010_012_007_004_006": ["010_shared002_lgb", "012_lgb010_bias", "007_lgb002_bias", "004_xgb_raw", "006_lgb_color_min"],
    "D_010_012_007_006": ["010_shared002_lgb", "012_lgb010_bias", "007_lgb002_bias", "006_lgb_color_min"],
    "E_012_007_006": ["012_lgb010_bias", "007_lgb002_bias", "006_lgb_color_min"],
    "F_010_012": ["010_shared002_lgb", "012_lgb010_bias"],
    "G_012_010_007": ["012_lgb010_bias", "010_shared002_lgb", "007_lgb002_bias"],
    "H_012_only": ["012_lgb010_bias"],
    "I_010_only": ["010_shared002_lgb"],
    "J_all_002_003_004_006_007_010_012": ["002_lgb_raw", "003_cat_raw", "004_xgb_raw", "006_lgb_color_min", "007_lgb002_bias", "010_shared002_lgb", "012_lgb010_bias"],
}
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)

OUTDIR: /kaggle/working/exp_20260605_013_blend_add012_lgb010_bias_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def load_npy_from_dataset(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing npy file: {path}\nEdit MODEL_SPECS if the filename differs.")
    return path

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw": mats.append(p)
        elif mode == "rank": mats.append(rank_normalize_matrix(p))
        elif mode == "logit": mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats.append(p); mats.append(logit_transform(p))
        elif mode == "raw_rank_logit":
            mats.append(p); mats.append(rank_normalize_matrix(p)); mats.append(logit_transform(p))
        else: raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    cur_score = balanced_accuracy_score(y_true, cur.argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score = cur_score
            best_w = w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0: continue
                    cand_w = cand_w / cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score = score
                        best_w = cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score = best_score
                w = best_w
                improved = True
                hist.append({"iter": len(hist), "score": float(cur_score), "weights": w.copy().tolist()})
    final = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    return w, final, float(cur_score), hist

def nm_optimize_weights(y_true, probas, maxiter=1200):
    n = len(probas)
    def unpack(theta):
        e = np.exp(theta - np.max(theta))
        return e / e.sum()
    def objective(theta):
        w = unpack(theta)
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))
    res = minimize(objective, np.zeros(n), method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-7, "fatol": 1e-10})
    w = unpack(res.x)
    p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    score = balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res

In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================
for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists(): raise FileNotFoundError(p)
if not NPY_DIR.exists(): raise FileNotFoundError(NPY_DIR)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)
le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy_from_dataset(spec["oof"])
    pred_path = load_npy_from_dataset(spec["pred"])
    oof_arr = np.load(oof_path).astype(np.float32)
    pred_arr = np.load(pred_path).astype(np.float32)
    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)
    oof[key] = oof_arr
    pred[key] = pred_arr
    row = dict(spec)
    row["resolved_oof_path"] = str(oof_path)
    row["resolved_pred_path"] = str(pred_path)
    row["oof_shape"] = list(oof_arr.shape)
    row["pred_shape"] = list(pred_arr.shape)
    resolved_specs.append(row)
model_keys = [s["key"] for s in MODEL_SPECS]
print("train:", train.shape)
print("test :", test.shape)
print("class_names:", class_names)
display(pd.DataFrame(resolved_specs))

train: (577347, 12)
test : (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']


,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,raw_reference,oof_lgb_strict_raw_proba.npy,pred_lgb_strict_raw_proba.npy,0.956906,0.95800,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,weak_reference,oof_cat_strict_raw_proba.npy,pred_cat_strict_raw_proba.npy,0.952458,0.95351,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,aux_reference,oof_xgb_strict_raw_proba.npy,pred_xgb_strict_raw_proba.npy,0.955784,0.95638,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,006_lgb_color_min,exp_20260603_006_lgb_color_index_min,LightGBM,blend_auxiliary,oof_lgb_color_index_min_proba.npy,pred_lgb_color_index_min_proba.npy,0.957046,0.95663,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
4,007_lgb002_bias,exp_20260603_007_lgb002_bias_search,LightGBM,old_mainline_auxiliary,oof_exp_20260603_007_lgb002_bias_search_proba.npy,pred_exp_20260603_007_lgb002_bias_search_proba...,0.958610,0.95921,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
5,010_shared002_lgb,exp_20260604_010_shared002_lgb_as_is,LightGBM,mainline_single_raw,oof_exp_20260604_010_shared002_lgb_as_is_proba...,pred_exp_20260604_010_shared002_lgb_as_is_prob...,0.962513,0.96357,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
6,012_lgb010_bias,exp_20260605_012_lgb010_bias_search,LightGBM,mainline_single_bias,oof_exp_20260605_012_lgb010_bias_search_proba.npy,pred_exp_20260605_012_lgb010_bias_search_proba...,0.963228,0.96466,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
# ============================================================
# 3. Individual scores
# ============================================================
rows = []
for spec in resolved_specs:
    key = spec["key"]
    y_pred = proba_to_pred(oof[key])
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)
individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
6,012_lgb010_bias,exp_20260605_012_lgb010_bias_search,LightGBM,mainline_single_bias,0.963228,0.96466,0.963228,0.0,0.964634,0.968944,0.956107
5,010_shared002_lgb,exp_20260604_010_shared002_lgb_as_is,LightGBM,mainline_single_raw,0.962513,0.96357,0.962513,0.0,0.967675,0.967407,0.952456
4,007_lgb002_bias,exp_20260603_007_lgb002_bias_search,LightGBM,old_mainline_auxiliary,0.958610,0.95921,0.958610,0.0,0.976693,0.966699,0.932438
3,006_lgb_color_min,exp_20260603_006_lgb_color_index_min,LightGBM,blend_auxiliary,0.957046,0.95663,0.957046,0.0,0.978145,0.964872,0.928122
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,raw_reference,0.956906,0.95800,0.956906,0.0,0.978431,0.965000,0.927288
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,aux_reference,0.955784,0.95638,0.955784,0.0,0.978582,0.963318,0.925451
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,weak_reference,0.952458,0.95351,0.952458,0.0,0.977432,0.963498,0.916445


In [5]:
# ============================================================
# 4. Pairwise OOF correlation / disagreement / error overlap
# ============================================================
pair_rows = []
pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}
for a, b in combinations(model_keys, 2):
    pa, pb = oof[a], oof[b]
    ya, yb = pred_labels[a], pred_labels[b]
    wrong_a, wrong_b = wrong_flags[a], wrong_flags[b]
    both = wrong_a & wrong_b
    either = wrong_a | wrong_b
    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(flatten_proba(pa), flatten_proba(pb)),
        "spearman_flat_proba": corr_spearman(flatten_proba(pa), flatten_proba(pb)),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both.mean()),
        "either_wrong_rate": float(either.mean()),
        "error_jaccard": float(both.sum() / max(either.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }
    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])
    pair_rows.append(row)
pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(flatten_proba(oof[a]), flatten_proba(oof[b]))
        spearman_flat.loc[a, b] = corr_spearman(flatten_proba(oof[a]), flatten_proba(oof[b]))
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))
pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")
display(pearson_flat)
display(argmax_agreement)

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
10,003_cat_raw,012_lgb010_bias,0.982871,0.960141,0.973557,0.026443,0.034134,0.035713,0.021905,0.047942,0.456917,0.012228,0.013808,0.977896,0.953054,0.990172,0.857740,0.964721,0.887030
17,006_lgb_color_min,012_lgb010_bias,0.983924,0.967404,0.975032,0.024968,0.031716,0.035713,0.021441,0.045988,0.466235,0.010275,0.014272,0.978775,0.957765,0.990739,0.862223,0.967243,0.919983
9,003_cat_raw,010_shared002_lgb,0.984122,0.963576,0.974947,0.025053,0.034134,0.034560,0.022018,0.046676,0.471723,0.012116,0.012542,0.979332,0.953033,0.990859,0.859204,0.966223,0.888231
14,004_xgb_raw,012_lgb010_bias,0.984131,0.959922,0.974991,0.025009,0.032128,0.035713,0.021633,0.046208,0.468176,0.010495,0.014080,0.979807,0.952284,0.989839,0.855149,0.967806,0.845036
5,002_lgb_raw,012_lgb010_bias,0.985160,0.968408,0.976068,0.023932,0.031622,0.035713,0.021898,0.045437,0.481950,0.009724,0.013815,0.980388,0.960026,0.991333,0.868627,0.970258,0.915225
16,006_lgb_color_min,010_shared002_lgb,0.985263,0.970892,0.976470,0.023530,0.031716,0.034560,0.021575,0.044701,0.482641,0.010141,0.012985,0.980378,0.957714,0.991498,0.865871,0.969025,0.923995
13,004_xgb_raw,010_shared002_lgb,0.985468,0.962224,0.976527,0.023473,0.032128,0.034560,0.021824,0.044864,0.486449,0.010304,0.012736,0.981403,0.952266,0.990590,0.855955,0.969588,0.849454
19,007_lgb002_bias,012_lgb010_bias,0.986056,0.969933,0.976851,0.023149,0.031676,0.035713,0.022312,0.045077,0.494986,0.009364,0.013401,0.981511,0.960025,0.991863,0.870006,0.971591,0.919587
4,002_lgb_raw,010_shared002_lgb,0.986522,0.971515,0.977527,0.022473,0.031622,0.034560,0.022047,0.044135,0.499549,0.009575,0.012512,0.982032,0.959982,0.992098,0.871675,0.972117,0.918931
18,007_lgb002_bias,010_shared002_lgb,0.987218,0.972312,0.978180,0.021820,0.031676,0.034560,0.022396,0.043840,0.510845,0.009280,0.012164,0.982911,0.959981,0.992502,0.872314,0.973169,0.922695


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias,010_shared002_lgb,012_lgb010_bias
002_lgb_raw,1.000000,0.996519,0.997944,0.998256,0.999904,0.986522,0.985160
003_cat_raw,0.996519,1.000000,0.997419,0.996427,0.996401,0.984122,0.982871
004_xgb_raw,0.997944,0.997419,1.000000,0.997113,0.997839,0.985468,0.984131
006_lgb_color_min,0.998256,0.996427,0.997113,1.000000,0.998172,0.985263,0.983924
007_lgb002_bias,0.999904,0.996401,0.997839,0.998172,1.000000,0.987218,0.986056
010_shared002_lgb,0.986522,0.984122,0.985468,0.985263,0.987218,1.000000,0.999812
012_lgb010_bias,0.985160,0.982871,0.984131,0.983924,0.986056,0.999812,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias,010_shared002_lgb,012_lgb010_bias
002_lgb_raw,1.000000,0.989538,0.992682,0.992457,0.997759,0.977527,0.976068
003_cat_raw,0.989538,1.000000,0.990425,0.989350,0.989230,0.974947,0.973557
004_xgb_raw,0.992682,0.990425,1.000000,0.990695,0.992239,0.976527,0.974991
006_lgb_color_min,0.992457,0.989350,0.990695,1.000000,0.992232,0.976470,0.975032
007_lgb002_bias,0.997759,0.989230,0.992239,0.992232,1.000000,0.978180,0.976851
010_shared002_lgb,0.977527,0.974947,0.976527,0.976470,0.978180,1.000000,0.997139
012_lgb010_bias,0.976068,0.973557,0.974991,0.975032,0.976851,0.997139,1.000000


In [6]:
# ============================================================
# 5. Blend diagnostics
# ============================================================
blend_rows = {}
blend_pred_outputs = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_pred_outputs[(set_name, method)] = test_p

for set_name, keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")
    probas = [oof[k] for k in keys]
    pred_probas = [pred[k] for k in keys]
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred, weights=np.ones(len(keys)) / len(keys))
    if len(keys) >= 2:
        hc_w, hc_oof, hc_score, hc_hist = hill_climb_weights(y, probas)
        hc_pred = normalize_rows(sum(hc_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=hc_w, extra={"hc_history_len": len(hc_hist)})
        nm_w, nm_oof, nm_score, nm_res = nm_optimize_weights(y, probas)
        nm_pred = normalize_rows(sum(nm_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=nm_w, extra={"nm_success": bool(nm_res.success), "nm_message": str(nm_res.message)})
    for mode in ["raw", "raw_logit", "raw_rank_logit", "logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=2000, random_state=SEED, n_jobs=-1)
            lr.fit(X_meta, y)
            lr_oof = lr.predict_proba(X_meta)
            lr_pred = lr.predict_proba(X_test_meta)
            record_blend(set_name, f"logreg_{mode}", keys, lr_oof, lr_pred, extra={"C": 0.05})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            rc_oof = softmax_np(rc.decision_function(X_meta), axis=1)
            rc_pred = softmax_np(rc.decision_function(X_test_meta), axis=1)
            record_blend(set_name, f"ridgeclf_{mode}", keys, rc_oof, rc_pred, extra={"alpha": 10.0})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False)
display(blend_df.head(120))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

===== A_011_best_recheck_010_007_004_006: ['010_shared002_lgb', '007_lgb002_bias', '004_xgb_raw', '006_lgb_color_min'] =====
[WARN] logreg failed: A_011_best_recheck_010_007_004_006 raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_011_best_recheck_010_007_004_006 raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_011_best_recheck_010_007_004_006 raw_rank_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_011_best_recheck_010_007_004_006 raw_rank_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] logreg failed: A_011_best_recheck_010_007_004_006 logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_011_best_recheck_010_007_004_006 logit: Input X contains infinity or a value too large for dtype('float32').
===== B_replace_010_with_012_012_007_004_006:

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,hc_history_len,nm_success,nm_message,C,alpha
11,C_keep_010_012_007_004_006,hc_nonnegative_raw,"010_shared002_lgb,012_lgb010_bias,007_lgb002_b...",5,0.964807,"{""010_shared002_lgb"": 0.0, ""012_lgb010_bias"": ...",0.971765,0.968833,0.953822,10.0,NaN,NaN,NaN,NaN
6,B_replace_010_with_012_012_007_004_006,hc_nonnegative_raw,"012_lgb010_bias,007_lgb002_bias,004_xgb_raw,00...",4,0.964787,"{""012_lgb010_bias"": 0.5631927848235014, ""007_l...",0.971863,0.968833,0.953665,10.0,NaN,NaN,NaN,NaN
27,E_012_007_006,hc_nonnegative_raw,"012_lgb010_bias,007_lgb002_bias,006_lgb_color_min",3,0.964782,"{""012_lgb010_bias"": 0.5600779963930564, ""007_l...",0.972001,0.968850,0.953496,7.0,NaN,NaN,NaN,NaN
16,D_010_012_007_006,hc_nonnegative_raw,"010_shared002_lgb,012_lgb010_bias,007_lgb002_b...",4,0.964780,"{""010_shared002_lgb"": 0.008816024304429317, ""0...",0.972144,0.968833,0.953363,8.0,NaN,NaN,NaN,NaN
78,J_all_002_003_004_006_007_010_012,hc_nonnegative_raw,"002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_co...",7,0.964746,"{""002_lgb_raw"": 0.03851926397615585, ""003_cat_...",0.972308,0.968773,0.953157,14.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,H_012_only,logreg_raw_rank_logit,012_lgb010_bias,1,0.955889,None,0.978025,0.960552,0.929090,NaN,NaN,NaN,0.05,NaN
44,F_010_012,logreg_raw_rank_logit,"010_shared002_lgb,012_lgb010_bias",2,0.955889,None,0.978060,0.960578,0.929029,NaN,NaN,NaN,0.05,NaN
36,E_012_007_006,ridgeclf_logit,"012_lgb010_bias,007_lgb002_bias,006_lgb_color_min",3,0.951462,None,0.979376,0.950104,0.924907,NaN,NaN,NaN,NaN,10.0
76,I_010_only,ridgeclf_logit,010_shared002_lgb,1,0.951259,None,0.978515,0.951188,0.924073,NaN,NaN,NaN,NaN,10.0


In [7]:
# ============================================================
# 6. Save best blend only
# ============================================================
# This intentionally saves only the best blend npy. 014 only needs this input.
best_row = blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")
print("Best blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method in ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]:
    probas = [oof[k] for k in best_keys]
    pred_probas = [pred[k] for k in best_keys]
    if best_method == "avg":
        w = np.ones(len(best_keys), dtype=float) / len(best_keys)
    else:
        weights_json = json.loads(best_row["weights_json"])
        w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * probas[i] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred_probas[i] for i in range(len(best_keys))))
else:
    raise RuntimeError(
        f"Best method is {best_method}. This notebook saves best npy only for avg/HC/NM. "
        "Review blend_diagnostics.csv before using linear meta as a final candidate."
    )

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)
best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / "oof_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend_proba.npy"
best_pred_npy = OUTDIR / "pred_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])
best_submission_path = OUTDIR / "submission_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
}
save_json(best_info, OUTDIR / "best_blend_info.json")
saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)

Best blend:
{
  "set_name": "C_keep_010_012_007_004_006",
  "method": "hc_nonnegative_raw",
  "keys": "010_shared002_lgb,012_lgb010_bias,007_lgb002_bias,004_xgb_raw,006_lgb_color_min",
  "n_models": 5,
  "balanced_accuracy": 0.9648069017992585,
  "weights_json": "{\"010_shared002_lgb\": 0.0, \"012_lgb010_bias\": 0.5771029948987743, \"007_lgb002_bias\": 0.17261102249029628, \"004_xgb_raw\": 0.07767496012063332, \"006_lgb_color_min\": 0.17261102249029628}",
  "recall_GALAXY": 0.9717653915439228,
  "recall_QSO": 0.9688329648378478,
  "recall_STAR": 0.953822349016005,
  "hc_history_len": 10.0,
  "nm_success": NaN,
  "nm_message": NaN,
  "C": NaN,
  "alpha": NaN
}
best_oof_score: 0.9648069017992585


,set_name,method,balanced_accuracy,submission,oof_proba,pred_proba
0,C_keep_010_012_007_004_006,hc_nonnegative_raw,0.964807,submission_exp_20260605_013_blend_add012_lgb01...,oof_exp_20260605_013_blend_add012_lgb010_bias_...,pred_exp_20260605_013_blend_add012_lgb010_bias...


In [8]:
# ============================================================
# 7. Save summary / memo
# ============================================================
def get_ind_score(key):
    return float(individual_df.loc[individual_df["key"] == key, "recomputed_oof_cv"].iloc[0])

judgement = {
    "current_reference_scores": {
        "011_best_blend_cv": 0.9640545915420509,
        "011_best_blend_public_lb": 0.96523,
        "010_raw_cv": 0.9625129421154345,
        "010_raw_public_lb": 0.96357,
        "012_bias_cv": 0.9632282943283176,
        "012_bias_public_lb": 0.96466,
    },
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_blend": best_info,
    "main_questions": {
        "replace_010_with_012": "Check B/E/H against A/I and 011 best reference.",
        "keep_010_and_012_together": "Check C/D/F/G against B and A.",
        "keep_007": "Check whether 007 retains nonzero HC/NM weight in best blend.",
        "keep_006": "Check whether 006 retains nonzero HC/NM weight in best blend.",
        "keep_004": "Previous 011 best gave 004 weight 0. Confirm whether it remains unnecessary.",
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Add 012 bias-searched 010 to 011 blend/correlation check",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add012_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 012 lgb010 bias search",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-05",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from the fixed npy-files Kaggle dataset folder, "
            "add 012 bias-searched 010 to the 011 blend/correlation table, "
            "and decide whether raw 010 should be replaced by 012 or both should remain. "
            "Save only the best blend OOF/pred proba for the next bias-search experiment."
        ),
        "success_criteria": [
            "load 002/003/004/006/007/010/012 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "evaluate predefined blend sets centered on 010/012",
            "save only best blend OOF/pred npy",
            "save best blend submission",
            "save memo and summary",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "corr_matrix_pearson_flat_proba": "corr_matrix_pearson_flat_proba.csv",
        "corr_matrix_spearman_flat_proba": "corr_matrix_spearman_flat_proba.csv",
        "matrix_argmax_agreement": "matrix_argmax_agreement.csv",
        "matrix_error_jaccard": "matrix_error_jaccard.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "best_blend_info": "best_blend_info.json",
        "saved_submission_candidates": "saved_submission_candidates.csv",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "role_judgement": "role_judgement.json",
        "summary": "blend_add012_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "012 improves raw 010 as a single model, but must be checked inside the 011 blend structure. "
            "The best 013 blend should be the sole input to 014 bias search."
        ),
        "next_action": [
            "Review blend_diagnostics.csv",
            "Review best_blend_info.json",
            "Submit best blend if it improves enough over 011 best candidate",
            "Use best blend OOF/pred npy as input to 014 bias search",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))
print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - best_blend_info.json
 - blend_add012_summary.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - oof_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pred_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend_proba.npy
 - role_judgement.json
 - saved_submission_candidates.csv
 - submission_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend.csv
